In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import logging

In [ ]:
import os
import sys

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.logger import setup_logging
setup_logging(log_dir='../logs')

logger = logging.getLogger('eda')
logger.info('EDA notebook started')

In [ ]:
train_data_path = r"..\data\raw\train.csv"
store_data_path = r"..\data\raw\store.csv"
test_data_path = r"..\data\raw\test.csv"
train = pd.read_csv(train_data_path, parse_dates=['Date'])
test = pd.read_csv(test_data_path, parse_dates=['Date'])
store = pd.read_csv(store_data_path)
pd.set_option('display.max_columns', None)
df = train.merge(store, on='Store', how='left')
df_test = test.merge(store, on='Store', how='left')
logger.info('Train shape: %s | Test shape: %s', df.shape, df_test.shape)

In [ ]:
df.head()

In [ ]:
df['DayOfWeek'].unique()

In [ ]:
print(f"Shape of data is:- {df.shape}")

In [ ]:
df.describe().T

In [ ]:
print("Data:-")
print(df.isnull().sum())

In [ ]:
print(f"No. of duplicate rows:- {df.duplicated().sum()}")
df = df.drop_duplicates()

In [ ]:
df.info()

In [ ]:
print(f"Data:- {df['PromoInterval'].value_counts()}")

In [ ]:
print(f"Data:- {df['Promo2'].value_counts()}")

In [ ]:
# Nan in Promo2SinceYear when Promo2 was '0'
print("Unique Values in Promo2SinceYear of Data Set when we have 'Zero' in Promo2")
print(df[df['Promo2'] == 0]['Promo2SinceYear'].value_counts(dropna=False))

In [ ]:
# Nan in Promo2SinceWeek when Promo2 was '0'
print("Unique Values in Promo2SinceWeek of Data Set when we have 'Zero' in Promo2")
print(df[df['Promo2'] == 0]['Promo2SinceWeek'].value_counts(dropna=False))

In [ ]:
# Nan in PromoInterval when Promo2 was '0'
print("Unique Values in PromoInterval of Data Set when we have 'Zero' in Promo2")
print(df[df['Promo2'] == 0]['PromoInterval'].value_counts(dropna=False))

- Since there were no "Promo2" so there are nan in 'Promo2SinceWeek' and 'Promo2SinceYear' and 'PromoInterval'
_________________________________________________________________________________________

In [ ]:
# Create a column nammed "Promo2SinceDays" which represent both Promo2SinceWeek and Promo2SinceYear
# Create Promo2StartDate
df['Promo2StartDate'] = pd.to_datetime(
    df['Promo2SinceYear'].fillna(0).astype(int).astype(str) +
    df['Promo2SinceWeek'].fillna(0).astype(int).astype(str) +
    '1',
    format='%G%V%u',
    errors='coerce'
)

# Create Promo2SinceDays
df['Promo2SinceDays'] = (
    df['Date'] - df['Promo2StartDate']
).dt.days

In [ ]:
# Since there were no Promo2 so fill '0' in Promo2SinceDays and PromoInterval

df['Promo2SinceDays'] = df['Promo2SinceDays'].fillna(0)
df['PromoInterval'] = df['PromoInterval'].fillna('no_promo2')

In [ ]:
df.columns

In [ ]:
df.head()

In [ ]:
# Fill the missing values in CompetitionDistance by "mean"

df['CompetitionDistance'] = df['CompetitionDistance'].fillna(df['CompetitionDistance'].mean())

In [ ]:
# Create a column called CompetitionOpenDay which represent both CompetitionOpenSinceYear and CompetitionOpenSinceMonth
df['CompetitionOpenDate'] = pd.to_datetime(
    dict(
        year=df['CompetitionOpenSinceYear'],
        month=df['CompetitionOpenSinceMonth'],
        day=1
    ),
    errors='coerce'
)
df['CompetitionOpenSinceDays'] = (
    df['Date'] - df['CompetitionOpenDate']
).dt.days

In [ ]:
# Fill the missing values in CompetationOpenSinceMonth & Year with '0' which represent not opened till now or information is missing

df['CompetitionOpenSinceDays'] = df['CompetitionOpenSinceDays'].fillna(0)

In [ ]:
# Drop the Unnecessary columns
df = df.drop(columns=['CompetitionOpenSinceYear', 'CompetitionOpenSinceMonth', 'CompetitionOpenDate', 'Promo2SinceYear', 'Promo2SinceWeek', 'Promo2StartDate'])

In [ ]:
df[df['Promo2SinceDays'] < 0].head()

In [ ]:
df[df['CompetitionOpenSinceDays'] < 0].head()

In [ ]:
"""
-> Since the competition which will opened in future, and promo which will be applied in future is mentioned here in the data,
-> So there are negative values in the CompetitionOpenSinceDays and Promo2SinceDays.
-> So if there are -ve values, it means no promo for now and no compitation for now. 
-> It means CompetitionOpenSinceDays is '0' and Promo2SinceDays is '0'
"""

# Convert negative values to 0

df['Promo2SinceDays'] = df['Promo2SinceDays'].clip(lower=0)
df['CompetitionOpenSinceDays'] = df['CompetitionOpenSinceDays'].clip(lower=0)

In [ ]:
plt.figure(figsize=(12, 5))

# Histogram + KDE
sns.histplot(df['Sales'], bins=50, kde=True)

plt.title("Distribution of Sales")
plt.xlabel("Sales")
plt.ylabel("Frequency")

plt.show()

In [ ]:
df['Sales'].skew()

In [ ]:
store_open_but_sales_zero = df[(df['Sales'] == 0) & (df['Open'] == 1)]
store_open_but_sales_zero.shape

In [ ]:
store_open_but_sales_zero['Store'].value_counts()

- There are 54 such stores which were open but still sales were 0.
- Store 28 has 3 such times, stores 835, 1039, 665, 623, 1017, 1100, 25, 102, 364, 339, 983 has 2 such times and rest had 1 such time
- [971, 674, 699, 708, 357, 227, 835, 835, 548, 28, 28, 28, 102, 238, 303, 387, 882, 887, 102, 925, 57, 1017, 1017, 1100, 1100, 661, 850, 986, 327, 25, 25, 623, 623, 983, 983, 663, 391, 927, 1039, 1039, 665, 665, 700, 681, 364, 364, 589, 948, 353, 259, 339, 339, 232, 762]

In [ ]:
import pandas as pd
import plotly.express as px

# Convert to datetime
store_open_but_sales_zero['Date'] = pd.to_datetime(
    store_open_but_sales_zero['Date']
)

# Count occurrences per day
daily_counts = (
    store_open_but_sales_zero
    .groupby('Date')
    .size()
    .reset_index(name='Count')
)

# Plot interactive line chart
fig = px.line(
    daily_counts,
    x='Date',
    y='Count',
    markers=True,
    title='Open Stores with Zero Sales Over Time'
)

# Improve layout
fig.update_layout(
    template='plotly_white',
    xaxis_title='Date',
    yaxis_title='Number of Stores',
    hovermode='x unified'
)

fig.show()

In [ ]:
num_col = []
cat_col = []
date_col = []
binary_col = []

for col in df.columns:
    
    # Datetime columns
    if pd.api.types.is_datetime64_any_dtype(df[col]):
        date_col.append(col)
    
    # Binary columns (only 2 unique values)
    elif df[col].nunique() == 2:
        binary_col.append(col)
    
    # Numerical columns
    elif pd.api.types.is_numeric_dtype(df[col]):
        num_col.append(col)
    
    # Categorical columns
    else:
        cat_col.append(col)

# Print results
print("Numerical Columns:")
print(num_col)

print("\nCategorical Columns:")
print(cat_col)

print("\nDatetime Columns:")
print(date_col)

print("\nBinary Columns:")
print(binary_col)

In [ ]:
cat_col = ['StateHoliday', 'StoreType', 'Assortment', 'PromoInterval', 'DayOfWeek']

In [ ]:
num_col = ['Sales', 'Customers', 'CompetitionDistance', 'CompetitionOpenSinceDays', 'Promo2SinceDays']

## <span style="color:red">Categorical Columns</span>

In [ ]:
df['StateHoliday'] = df['StateHoliday'].astype(str)

In [ ]:
df['StateHoliday'].value_counts()

#### <span style="color:pink">Univariate Analysis</span>

In [ ]:
df.columns

In [ ]:
# Loop through each categorical column
for col in cat_col:
    
    # Value counts
    value_counts = (
        df[col]
        .value_counts()
        .reset_index()
    )
    
    value_counts.columns = [col, 'Count']
    
    # Create interactive bar plot
    fig = px.bar(
        value_counts,
        x=col,
        y='Count',
        text='Count',
        title=f'Value Counts of {col}'
    )
    
    # Improve layout
    fig.update_traces(
        textposition='outside'
    )
    
    fig.update_layout(
        template='plotly_white',
        xaxis_title=col,
        yaxis_title='Count'
    )
    
    fig.show()

#### <span style="color:pink">Bivariate Analysis</span>

In [ ]:
for col in cat_col:
    
    plt.figure(figsize=(8,5))
    
    ax = sns.barplot(
        data=df,
        x=col,
        y='Sales',
        estimator='mean'
    )
    
    # Add values on bars
    for p in ax.patches:
        height = p.get_height()
        
        ax.annotate(
            f'{height:,.0f}',                      # format number
            (p.get_x() + p.get_width() / 2., height),
            ha='center',
            va='bottom',
            fontsize=10,
            xytext=(0, 5),
            textcoords='offset points'
        )
    
    plt.title(f'Average Sales by {col}')
    plt.ylabel('Average Sales')
    plt.xlabel(col)
    
    plt.tight_layout()
    plt.show()

## 📌 Key Business Insights

### 🏖️ Impact of State Holidays on Sales
- During **State Holidays** (`a`, `b`, `c`), both **Average Sales** and **Total Sales** drop significantly compared to normal working days.
- This indicates that customer activity is extremely low during holiday periods.

---

### 🏪 Store Type Performance Analysis
- The frequency of **Store Type `b`** is much lower compared to other store types.
- However, despite having fewer stores, **Store Type `b` generates nearly 1.7× higher average sales** than other store categories.
- In terms of average sales performance, the remaining store types show relatively similar behavior.

---

### 🛒 Assortment Analysis
#### 📊 Value Count Distribution
```text
a > c > b
---

###  Week Day Sales Analysis
- The frequency of every weekdays are almost same but  **Monday** has higher average sales compared to other days.
- **Saturday** perform just a bit lower then other weekday.
- **Sunday's** performance is very poor


In [ ]:
df['sales_per_cus'] = (
    df['Sales'] / df['Customers']
)

df['sales_per_cus'] = (
    df['sales_per_cus']
    .replace([float('inf'), -float('inf')], 0)
    .fillna(0)
)

In [ ]:
for col in cat_col:
    
    plt.figure(figsize=(8,5))
    
    ax = sns.barplot(
        data=df,
        x=col,
        y='sales_per_cus',
        estimator='mean'
    )
    
    # Add values on bars
    for p in ax.patches:
        height = p.get_height()
        
        ax.annotate(
            f'{height:,.2f}',                      # format number
            (p.get_x() + p.get_width() / 2., height),
            ha='center',
            va='bottom',
            fontsize=10,
            xytext=(0, 5),
            textcoords='offset points'
        )
    
    plt.title(f'Sales by Customers Ratio {col}')
    plt.ylabel('Sales by Customers Ratio')
    plt.xlabel(col)
    
    plt.tight_layout()
    plt.show()

## 📌 Customer Purchase Behavior Insights

### 🏖️ State Holiday Impact on Sales per Customer
- During **State Holidays**, the **Sales per Customer ratio** is lower compared to normal working days.
- Among all holiday categories, **State Holiday `a`** performs comparatively better than `b` and `c`.
- This indicates that customer spending reduces during holidays, although some holiday types still maintain moderate purchasing activity.

---

### 🏪 Store Type Analysis (Sales per Customer)
- **Store Type `d`** shows a significantly higher **Sales per Customer ratio** compared to other store types.
- This may indicate that these stores are selling:
  - Higher-priced products
  - Bulk items
  - Premium category products

- On the other hand, **Store Type `b`** has a comparatively lower Sales per Customer ratio.
- This suggests that these stores may primarily sell:
  - Low-cost products
  - Daily-use essentials
  - Small-ticket items

---

### 🛒 Assortment-wise Sales per Customer
#### 📈 Performance Ranking
```text
c > a > b
---

###  Week Day Sales Per Customer Analysis
- The frequency of every weekdays are almost same but  **Monday** has higher Sales per customer ratio compared to other days.
- **Saturday** perform as good as other weekday.
- **Sunday's** performance is very poor

In [ ]:
df.groupby('StoreType')['Assortment'].count()

In [ ]:
cross_tab = pd.crosstab(df['StoreType'], df['Assortment'])
cross_tab

- Assortment 'b' is only used by StoreType 'b'

In [ ]:
pd.crosstab(df['StoreType'], df['PromoInterval'])

- All the StoreType uses majorly PromoInterval as "Jan,Apr,Jul,Oct".

## <span style="color:red">Numerical Columns</span>

In [ ]:
num_col

### <span style="color:pink">Univariate Analysis</span>

In [ ]:
df[num_col].describe().T

In [ ]:
for col in num_col:
    
    plt.figure(figsize=(8,5))
    
    sns.histplot(
        df[col],
        kde=True,
        bins=30
    )
    
    plt.title(f'Distribution of {col}')
    
    plt.tight_layout()
    plt.show()

In [ ]:
sales_corr = (
    df[num_col]
    .corr()
    .loc[:, 'Sales']
    .sort_values(ascending=False)
)

print(sales_corr)

In [ ]:
sales_corr.drop('Sales').plot(kind='bar', figsize=(12,5))

plt.title('Correlation with Sales')
plt.ylabel('Correlation')
plt.xticks(rotation=45)
plt.show()

In [ ]:
for col in num_col:
    
    plt.figure(figsize=(6,4))
    
    sns.scatterplot(
        x=df[col],
        y=df['Sales']
    )
    
    plt.title(f'Sales vs {col}')
    plt.show()

In [ ]:
fig = px.box(
    df[num_col],
    template="plotly_white",
    title="Box Plot for Numerical Features"
)

fig.update_layout(
    height=700,
    width=1200,
    title_x=0.5
)

fig.show()

## <span style="color:red">Binary Flags</span>

#### <span style="color:pink">Univariate Analysis</span>

In [ ]:
imbalance_summary = pd.DataFrame(columns=['Column', 'Class_0', 'Class_1', 'Class_0_%', 'Class_1_%'])

for col in binary_col:
    counts = df[col].value_counts()
    percentages = df[col].value_counts(normalize=True) * 100

    imbalance_summary.loc[len(imbalance_summary)] = [
        col,
        counts.get(0, 0),
        counts.get(1, 0),
        round(percentages.get(0, 0), 2),
        round(percentages.get(1, 0), 2)
    ]

print(imbalance_summary)

## Class Imbalance Analysis of Binary Flag Columns

The dataset contains four binary flag features: `Open`, `Promo`, `SchoolHoliday`, and `Promo2`.  
The class distribution analysis shows varying levels of imbalance across these features.

| Column | Class 0 (%) | Class 1 (%) | Observation |
|---|---|---|---|
| Open | 16.99% | 83.01% | Highly imbalanced towards Class 1 |
| Promo | 61.85% | 38.15% | Moderately imbalanced |
| SchoolHoliday | 82.14% | 17.86% | Highly imbalanced towards Class 0 |
| Promo2 | 49.94% | 50.06% | Perfectly balanced |

### Key Observations

- **Open**:
  - Majority of stores are in the open state (`83.01%`).
  - The closed state (`16.99%`) represents a minority class.

- **Promo**:
  - Promotional events are moderately balanced, though non-promo days are more frequent.

- **SchoolHoliday**:
  - Strong imbalance exists, with most records corresponding to non-school holidays.

- **Promo2**:
  - This feature is almost perfectly balanced between both classes.

### Conclusion

The dataset contains both balanced and imbalanced binary features.  
Columns such as `Open` and `SchoolHoliday` exhibit significant class imbalance, which may influence machine learning model performance and should be considered during preprocessing or model training.

#### <span style="color:pink">Bivariate Analysis</span>

In [ ]:
# Create subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes = axes.flatten()

for i, col in enumerate(binary_col):

    # Average sales for each class
    avg_sales = df.groupby(col)['Sales'].mean()

    # Plot
    avg_sales.plot(
        kind='bar',
        ax=axes[i]
    )

    axes[i].set_title(f'Average Sales by {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Average Sales')
    axes[i].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## <span style="color:red">Date Time Column</span>

In [ ]:
# Group daily sales
daily_sales = (
    df.groupby('Date')['Sales']
    .sum()
    .reset_index()
)
# Interactive line plot
fig = px.line(
    daily_sales,
    x='Date',
    y='Sales',
    title='Total Sales Per Day'
)

fig.show()

In [ ]:
# Select only numerical columns
numeric_df = df.select_dtypes(include=['int64', 'float64'])

# Compute correlation matrix
corr_matrix = numeric_df.corr()

# Plot heatmap
plt.figure(figsize=(18, 12))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    linewidths=0.5
)

plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()